# NB05 — LIME Cross-Validation & IG-LIME Agreement
**Project:** CNN vs ViT Localization Faithfulness in Chest X-Ray
**Plan version:** v7.1 — **Notebook version: v7.1-5f** (all audit fixes applied)
**Stage 4b:** Run after NB04 is complete and `ig_manifest.csv` exists.

---

### What this notebook does
1. Mounts Drive and loads config + LoRA rank (thresholds.json removed — not used by LIME).
2. Loads `ig_manifest.csv` produced by NB04; asserts all 3 models present.
3. Loads all 3 model checkpoints one at a time (DenseNet121, ConvNeXtV2-Tiny, Swin-B+LoRA).
4. Runs LIME (`lime==0.2.0.1`, quickshift `kernel_size=4, max_dist=10, ratio=0.5`, `n_samples=1000`).
5. `predict_fn` returns **2-class output** `[P(not-target), P(target)]` — one fn per row — so LIME fits its surrogate only against the single target label (same scope as IG).
6. Saves LIME explanations to `lime_maps/{image_id}_{model}_lime.pkl` — resume-safe.
7. Computes IoU between binarised top-50% IG mask and equivalent-area LIME mask.
8. `healthy` rows (no consensus box / no top50 mask) get `iou_valid=False` + `lime_ig_iou=NaN` for NB07 FP analysis.
9. Saves `results/lime_ig_agreement.csv`, `lime_manifest.csv`, `lime_agreement_summary.csv`.

### Outputs
```
lime_maps/
    {image_id}_{model}_lime.pkl

results/
    lime_manifest.csv
    lime_ig_agreement.csv          (NaN rows kept for NB07 FP analysis)
    lime_agreement_summary.csv     (pathological rows only → Table 2)
```


### Pre-flight checklist
- [ ] Runtime → Change runtime type → **T4 GPU**
- [ ] NB04 complete: `ig_manifest.csv` and all `_top50.npy` masks exist
- [ ] NB02 complete: all 3 `.pt` checkpoints, `lora_sweep.csv`


## Cell 1 — Drive mount

In [1]:
from google.colab import drive
drive.mount('/content/drive')

GDRIVE_ROOT = '/content/drive/MyDrive/cxr_faithfulness'
exec(open(f'{GDRIVE_ROOT}/config/startup.py').read())


Mounted at /content/drive
⏳ Installing strictly pinned architecture packages onto Colab's native stack (~30s)...
✅ Packages ready! Using native modern PyTorch and NumPy.


## Cell 2 — Pinned dependencies

In [2]:
# scikit-image pinned to 0.22.0 (quickshift random_seed param added in 0.19;
# uncontrolled upgrades could silently change segmentation results)
# !pip install -q \
#     torch==2.2.1 \
#     torchvision==0.17.1 \
#     timm==0.9.12 \
#     peft==0.6.2 \
#     scikit-learn==1.3.2 \
#     lime==0.2.0.1 \
#     scikit-image==0.22.0 \
#     opencv-python-headless || exit 1

import torch
assert torch.cuda.is_available(), 'GPU not available — Runtime → Change runtime type → GPU'
print(f'✅ GPU : {torch.cuda.get_device_name(0)}')
print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'   torch: {torch.__version__}')


✅ GPU : Tesla T4
   VRAM: 15.6 GB
   torch: 2.11.0+cu128


## Cell 3 — Imports, paths, constants

In [3]:
import gc, json, pickle, random, warnings
import numpy as np
import pandas as pd
from pathlib import Path

import cv2
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as tvmodels
import timm

# torch.cuda.amp.autocast() — torch.amp.autocast('cuda') only stable in ≥2.2.
# This notebook pins torch==2.2.1, so use the cuda-specific namespace.
from torch.cuda.amp import autocast

from peft import LoraConfig, get_peft_model
from lime.lime_image import LimeImageExplainer
from skimage.segmentation import quickshift

ROOT        = Path(GDRIVE_ROOT)
IMAGES_PATH = ROOT / 'data' / 'processed' / 'images'
MODELS_PATH = ROOT / 'models'
RESULTS_PATH= ROOT / 'results'
IG_PATH     = ROOT / 'ig_maps'
LIME_PATH   = ROOT / 'lime_maps'

RESULTS_PATH.mkdir(parents=True, exist_ok=True)
LIME_PATH.mkdir(parents=True, exist_ok=True)

IMG_SIZE          = 224
NUM_CLASSES       = 14
RANDOM_SEED       = 42
LIME_SAMPLES      = 1000
SAVE_EVERY        = 100
AGREEMENT_IOU_THR = 0.15   # threshold for agree_rate_gte15
LOW_AGREE_THRESH  = 0.60   # flag combos below this in Discussion

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

MODEL_NAMES = ['densenet121', 'convnextv2_tiny', 'swinb_lora']

print('✅ Imports and paths ready.')


✅ Imports and paths ready.


## Cell 4 — Reproducibility seed

In [4]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(RANDOM_SEED)
print(f'✅ Seed set: {RANDOM_SEED}')


✅ Seed set: 42


## Cell 5 — Load LoRA rank  

In [5]:
# LIME's predict_fn outputs raw sigmoid probabilities; F1-optimal thresholds are only needed at inference/evaluation time (NB06). Removed entirely.

sweep_path = MODELS_PATH / 'lora_sweep.csv'
assert sweep_path.exists(), f'❌ lora_sweep.csv not found: {sweep_path}. Run NB02 first.'
sweep_df = pd.read_csv(str(sweep_path))
assert 'rank'    in sweep_df.columns, f"'rank' column missing. Got: {sweep_df.columns.tolist()}"
assert 'val_auc' in sweep_df.columns, f"'val_auc' column missing. Got: {sweep_df.columns.tolist()}"
selected_rank = int(sweep_df.loc[sweep_df['val_auc'].idxmax(), 'rank'])
print(f'✅ LoRA rank: r{selected_rank}')


✅ LoRA rank: r32


## Cell 6 — Load IG manifest  

In [6]:
ig_manifest_path = RESULTS_PATH / 'ig_manifest.csv'
assert ig_manifest_path.exists(), f'❌ ig_manifest.csv not found: {ig_manifest_path}. Run NB04 first.'

ig_manifest_df = pd.read_csv(str(ig_manifest_path))

REQUIRED_COLS = ['image_id', 'model', 'subset', 'target_class', 'target_idx', 'ig_path', 'top50_path']
for col in REQUIRED_COLS:
    assert col in ig_manifest_df.columns, (
        f"❌ Column '{col}' missing from ig_manifest.csv. "
        f"Found: {ig_manifest_df.columns.tolist()}"
    )

#hard assert that ALL three planned models are present before any processing.
#  If NB04 timed out after one model, this catches it immediately.
present_models = set(ig_manifest_df['model'].unique())
for mn in MODEL_NAMES:
    assert mn in present_models, (
        f"❌ Model '{mn}' missing from ig_manifest.csv. "
        f"NB04 may have timed out. Present: {sorted(present_models)}"
    )

# removed the vectorised Path.exists() sweep across all rows at startup.
#  On 13 k+ rows that's ~13,500 Drive stat() calls before a single image is
#  processed.  Cell 12 already guards per-row with an assertion.
# (We only report the count so the user is aware of any gaps.)
print(f'✅ ig_manifest.csv loaded : {len(ig_manifest_df)} rows')
print(f'   Models   : {sorted(present_models)}')
print(f'   Subsets  : {ig_manifest_df["subset"].unique().tolist()}')
print(f'   Pathologies: {ig_manifest_df["target_class"].nunique()} unique classes')


✅ ig_manifest.csv loaded : 1528 rows
   Models   : ['convnextv2_tiny', 'densenet121', 'swinb_lora']
   Subsets  : ['patho', 'healthy']
   Pathologies: 11 unique classes


## Cell 7 — Inference transform and image loader

In [7]:
# Resize intentionally OMITTED: LIME passes images that were already resized to
# 224×224 by cv2.resize() in Cell 11 before explain_instance() is called.
# Keeping Resize here would be a harmless no-op, but removing it avoids
# the PIL round-trip cost on every one of the 1000 LIME perturbation images.
# If inference_transform is ever reused outside LIME (e.g. single-image debug),
# callers are responsible for pre-resizing to IMG_SIZE.
inference_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def load_image_rgb(image_id: str) -> np.ndarray:
    """Load PNG as uint8 RGB (H, W, 3). Raises AssertionError on missing file."""
    p = IMAGES_PATH / f'{image_id}.png'
    img = cv2.imread(str(p))
    assert img is not None, f'❌ Image not found: {p}. Run NB01 first.'
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

print('✅ Inference transform and image loader defined.')


✅ Inference transform and image loader defined.


## Cell 8 — Model loader  

In [8]:
def load_model(model_name: str, selected_rank: int, num_classes: int) -> nn.Module:
    """
    Builds and loads a fine-tuned checkpoint.

    strict_mode=True for CNN models — any key mismatch is a hard error.
    For swinb_lora: strict=False is used (PEFT adds adapter key prefixes), but a
    whitelist check ensures only known LoRA/adapter prefixes are missing — any
    unexpected backbone key absence is a hard error (FIX-E).
    """
    if model_name == 'densenet121':
        m = tvmodels.densenet121(weights=None)
        m.classifier = nn.Linear(m.classifier.in_features, num_classes)
        strict_mode = True

    elif model_name == 'convnextv2_tiny':
        # Conservative loader: instantiate the full timm classifier head and let
        # the checkpoint overwrite all head parameters (including any head.norm
        # state if present). This minimizes strict-load fragility across timm
        # head implementations while preserving the same 14-class architecture.
        m = timm.create_model(
            'convnextv2_tiny.fcmae_ft_in22k_in1k',
            pretrained=False,
            num_classes=num_classes,
        )
        strict_mode = True

    elif model_name == 'swinb_lora':
        base = timm.create_model(
            'swin_base_patch4_window7_224',
            pretrained=False,
            num_classes=num_classes,
        )
        # target_modules match NB02/NB03/NB04 contract — do not change
        lora_cfg = LoraConfig(
            r=selected_rank,
            lora_alpha=selected_rank * 2,
            target_modules=['qkv', 'proj'],
            lora_dropout=0.1,
            bias='none',
        )
        m = get_peft_model(base, lora_cfg)
        strict_mode = False    # PEFT adapter keys have different prefixes

    else:
        raise ValueError(f'Unknown model_name: {model_name}')

    ckpt_path = MODELS_PATH / f'{model_name}_finetuned.pt'
    assert ckpt_path.exists(), f'❌ Checkpoint not found: {ckpt_path}. Run NB02 first.'
    state = torch.load(str(ckpt_path), map_location='cpu', weights_only=True)
    res   = m.load_state_dict(state, strict=strict_mode)

    if strict_mode:
        assert len(res.missing_keys) == 0 and len(res.unexpected_keys) == 0, (
            f'❌ {model_name} state dict mismatch.\n'
            f'   Missing   : {res.missing_keys[:5]}\n'
            f'   Unexpected: {res.unexpected_keys[:5]}'
        )
    else:
        # for swinb_lora, only base_model.* / lora_* / base_layer.*
        # prefixes are allowed to be missing (added by PEFT at save time).
        # Any other missing key means the backbone was not fully saved → hard error.
        ALLOWED_MISSING_PREFIXES = ('base_model.', 'lora_', 'base_layer.')
        bad_missing = [
            k for k in res.missing_keys
            if not any(k.startswith(pfx) for pfx in ALLOWED_MISSING_PREFIXES)
        ]
        assert len(bad_missing) == 0, (
            f'❌ {model_name}: unexpected backbone keys missing from checkpoint — '
            f'checkpoint may be corrupt or incomplete.\n'
            f'   Bad missing keys (first 10): {bad_missing[:10]}'
        )
        print(f'   {model_name}: {len(res.missing_keys)} missing (all LoRA/adapter prefixes ✅), '
              f'{len(res.unexpected_keys)} unexpected')

    m = m.cuda().eval()
    print(f'✅ {model_name} loaded from {ckpt_path.name}')
    return m

print('✅ Model loader defined.')


✅ Model loader defined.


## Cell 9 — LIME prediction function factory  

In [9]:
def make_lime_predict_fn(model: nn.Module, target_idx: int):
    """
    Returns a predict_fn for LimeImageExplainer that outputs a 2-column array
    [P(not-target), P(target)] instead of the full 14-class vector.

    torch.cuda.amp.autocast() is used instead of torch.amp.autocast('cuda').
    The string-device form only stabilised in PyTorch ≥2.2; this notebook pins
    torch==2.2.1.

    LIME passes (N, H, W, 3) arrays — dtype may be float64 in [0, 255].
    np.clip + astype(uint8) prevents silent all-black batches.
    No cv2.resize here — images arrive at 224×224 from explain_instance.
    """
    def predict_fn(images: np.ndarray) -> np.ndarray:
        batch = []
        for img in images:
            img_uint8 = np.clip(img, 0, 255).astype(np.uint8)
            batch.append(inference_transform(img_uint8))
        x = torch.stack(batch).cuda()
        model.eval()
        with torch.inference_mode():
            with autocast():                          # FIX-B: cuda-specific namespace
                logits = model(x)                     # (N, 14)
        probs_target = torch.sigmoid(logits[:, target_idx]).cpu().numpy()  # (N,)
        # Return [P(not-target), P(target)] — standard LIME binary format
        return np.column_stack([1.0 - probs_target, probs_target])         # (N, 2)

    return predict_fn

print('✅ LIME prediction function factory defined.')
print('   Output shape per batch: (N, 2)  →  [P(not-target), P(target)]')


✅ LIME prediction function factory defined.
   Output shape per batch: (N, 2)  →  [P(not-target), P(target)]


## Cell 10 — LIME mask and IoU helpers  (FIX-H: fallback removed | FIX-K: mask squeeze)

In [10]:
def get_lime_mask(
    explanation,
    target_idx: int,
    ig_top50_mask: np.ndarray,
) -> np.ndarray | None:
    """
    Build a binary LIME mask matching the pixel count of ig_top50_mask.

    Returns None (→ iou_valid=False, NaN) when target_idx is absent from
    local_exp — LIME completely failed to attribute any mass to the target
    class. Returning 0.0 IoU would be scientifically wrong because it would
    look like 'total disagreement' rather than 'missing explanation'.


    Superpixel loop fills to target area regardless of weight sign — guarantees
    fair IoU comparison even for diffuse pathologies (FIX v1-#5 preserved).
    """
    # Verify mask is binary {0,1} — catches NB04 masks saved as {0,255}
    unique_vals = np.unique(ig_top50_mask)
    assert set(unique_vals).issubset({0, 1}), (
        f'❌ ig_top50_mask is not binary. Unique values: {unique_vals}. '
        f'NB04 may have saved the mask as 0/255 instead of 0/1.'
    )

    target_area = int(ig_top50_mask.sum())
    if target_area == 0:
        return np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)

    local_exp = explanation.local_exp

    # FIX-H: no fallback — absent target_idx → caller records NaN + iou_valid=False
    if target_idx not in local_exp:
        return None

    superpixel_weights = dict(local_exp[target_idx])
    segments = explanation.segments  # (224, 224) int

    # Sort descending by weight — include negative-weight superpixels
    sorted_sp = sorted(superpixel_weights.items(), key=lambda kv: kv[1], reverse=True)

    lime_mask   = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    accumulated = 0
    for sp_id, _ in sorted_sp:
        region          = (segments == sp_id)
        lime_mask[region] = 1
        accumulated    += int(region.sum())
        if accumulated >= target_area:
            break

    return lime_mask


def compute_iou(mask_a: np.ndarray, mask_b: np.ndarray) -> float:
    """Binary IoU between two (H, W) uint8 masks."""
    inter = np.logical_and(mask_a, mask_b).sum()
    uni   = np.logical_or(mask_a, mask_b).sum()
    return 0.0 if uni == 0 else float(inter / uni)


print('✅ LIME mask and IoU helpers defined.')


✅ LIME mask and IoU helpers defined.


## Cell 11 — Phase A: LIME generation — resume-safe  (FIX-F: predict_fn per row | FIX-G: quickshift params)

In [12]:
set_seed(RANDOM_SEED)

# Helper function to create a segmentation function with a fixed random seed
def make_seg_fn(seed=RANDOM_SEED):
    def seg_fn(x):
        np.random.seed(seed) # Seed numpy's global state for reproducibility
        return quickshift(x, kernel_size=4, max_dist=10, ratio=0.5)
    return seg_fn

lime_manifest_rows_temp = [] # Use a temporary list for initial population
fallback_counter   = {'count': 0}

lime_manifest_partial = RESULTS_PATH / 'lime_manifest_partial.csv'

# ── Seed dedup set from partial CSV AND populate lime_manifest_rows ───────
done_lime_keys = set()
if lime_manifest_partial.exists():
    try:
        _dfp = pd.read_csv(str(lime_manifest_partial))
        if not _dfp.empty: # Explicitly check if DataFrame is not empty after reading
            lime_manifest_rows_temp.extend(_dfp.to_dict('records'))
            done_lime_keys.update(set(zip(_dfp['image_id'], _dfp['model'])))
            print(f'Resumed lime_manifest from partial CSV: {len(lime_manifest_rows_temp)} rows')
        else:
            print('lime_manifest_partial.csv exists but is empty (only header or no data rows).')
    except pd.errors.EmptyDataError:
        print('lime_manifest_partial.csv exists but contains no columns to parse (EmptyDataError).')
    except Exception as e:
        print(f'An unexpected error occurred while reading lime_manifest_partial.csv: {e}')
else:
    print('No lime_manifest_partial.csv found.')

# Now, iterate through ig_manifest_df to find items that have .pkl files
# but were not in the partial CSV (e.g., if a crash occurred before saving CSV)
newly_found_pkls = []
for idx, row_ig in ig_manifest_df.iterrows():
    img_id     = row_ig['image_id']
    model_name = row_ig['model']
    key        = (img_id, model_name)
    lime_pkl_path = LIME_PATH / f'{img_id}_{model_name}_lime.pkl'

    if key not in done_lime_keys and lime_pkl_path.exists():
        newly_found_pkls.append({
            'image_id'     : img_id,
            'model'        : model_name,
            'subset'       : row_ig['subset'],
            'target_class' : row_ig['target_class'],
            'target_idx'   : int(row_ig['target_idx']),
            'lime_pkl_path': str(lime_pkl_path),
        })
        done_lime_keys.add(key)

if newly_found_pkls:
    lime_manifest_rows_temp.extend(newly_found_pkls)
    print(f'Added {len(newly_found_pkls)} manifest rows from existing .pkl files not in partial CSV.')

lime_manifest_rows = lime_manifest_rows_temp # Assign the fully populated list

print(f'✅ Resume state: {len(done_lime_keys)} images already processed (total in manifest: {len(lime_manifest_rows)})')

# ── Per-model generation loop ────────────────────────────────────────────────
for model_name in MODEL_NAMES:
    print(f'\n{"="*60}')
    print(f'LIME generation — {model_name}')
    print(f'{"="*60}')

    model = load_model(model_name, selected_rank, NUM_CLASSES)

    # FIX-F: explainer instantiated ONCE per model (not per row); predict_fn
    #        created PER ROW to isolate the 2-class output to the correct target.
    explainer = LimeImageExplainer(random_state=RANDOM_SEED)

    model_rows = ig_manifest_df[ig_manifest_df['model'] == model_name].to_dict('records')
    n_new = 0
    n_pkl_reused = 0

    for i, row in enumerate(model_rows):
        image_id   = row['image_id']
        target_idx = int(row['target_idx'])
        key        = (image_id, model_name)

        lime_pkl_path = LIME_PATH / f'{image_id}_{model_name}_lime.pkl'

        # ── Load or generate LIME explanation ──────────────────────────────
        if lime_pkl_path.exists():
            with open(str(lime_pkl_path), 'rb') as fh:
                explanation = pickle.load(fh)
            # Only increment reused if it was part of the original set that existed
            if key in done_lime_keys:
                n_pkl_reused += 1
        else:
            img_rgb     = load_image_rgb(image_id)
            img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE),
                                     interpolation=cv2.INTER_AREA)  # uint8 (H,W,3)

            # FIX-F: new predict_fn scoped to this row's target_idx
            predict_fn = make_lime_predict_fn(model, target_idx)

            explanation = explainer.explain_instance(
                img_resized,
                predict_fn,
                # labels=[1] because predict_fn column 1 == P(target)
                top_labels=None,
                labels=[1],
                hide_color=0,
                num_samples=LIME_SAMPLES,
                # FIX-G: max_dist=200→10, ratio=0.2→0.5
                # max_dist=200 on a 224px image spans ~89% of the diagonal,
                # merging most of the lung field into a few coarse superpixels.
                # max_dist=10 with ratio=0.5 produces anatomically meaningful
                # boundaries for low-contrast CXR pathologies.
                segmentation_fn=make_seg_fn(RANDOM_SEED)
            )
            with open(str(lime_pkl_path), 'wb') as fh:
                pickle.dump(explanation, fh, protocol=pickle.DEFAULT_PROTOCOL)
            n_new += 1

        # ── Append manifest row (dedup-guarded) ────────────────────────────
        if key not in done_lime_keys:
            lime_manifest_rows.append({
                'image_id'     : image_id,
                'model'        : model_name,
                'subset'       : row['subset'],
                'target_class' : row['target_class'],
                'target_idx'   : target_idx,
                'lime_pkl_path': str(lime_pkl_path),
            })
            done_lime_keys.add(key)

        # ── Incremental checkpoint ──────────────────────────────────────────
        if (i + 1) % SAVE_EVERY == 0:
            pd.DataFrame(lime_manifest_rows).to_csv(str(lime_manifest_partial), index=False)
            print(f'  [{i+1}] checkpoint saved ({n_new} new / {n_pkl_reused} reused)...')

    # Final partial save after each model (covers last incomplete batch)
    pd.DataFrame(lime_manifest_rows).to_csv(str(lime_manifest_partial), index=False)
    print(f'✅ {model_name}: {n_new} new, {n_pkl_reused} pkl-reused.')

    del model
    gc.collect()
    torch.cuda.empty_cache()

print(f'\n✅ Phase A complete. Manifest rows: {len(lime_manifest_rows)}')
# Note: fallback_counter is incremented in Phase B (Cell 12), not here.
# Phase A only generates and saves pkl files — mask/label checks happen in Phase B.

No lime_manifest_partial.csv found.
Added 1016 manifest rows from existing .pkl files not in partial CSV.
✅ Resume state: 1016 images already processed (total in manifest: 1016)

LIME generation — densenet121
✅ densenet121 loaded from densenet121_finetuned.pt
  [100] checkpoint saved (0 new / 100 reused)...
  [200] checkpoint saved (0 new / 200 reused)...
  [300] checkpoint saved (0 new / 300 reused)...
  [400] checkpoint saved (0 new / 400 reused)...
  [500] checkpoint saved (0 new / 500 reused)...
✅ densenet121: 0 new, 514 pkl-reused.

LIME generation — convnextv2_tiny
✅ convnextv2_tiny loaded from convnextv2_tiny_finetuned.pt


  0%|          | 0/1000 [00:00<?, ?it/s]

/tmp/ipykernel_2423/1184543510.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():                          # FIX-B: cuda-specific namespace


  0%|          | 0/1000 [00:00<?, ?it/s]

/tmp/ipykernel_2423/1184543510.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():                          # FIX-B: cuda-specific namespace


...

  0%|          | 0/1000 [00:00<?, ?it/s]

/tmp/ipykernel_2423/1184543510.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():                          # FIX-B: cuda-specific namespace


✅ convnextv2_tiny: 512 new, 0 pkl-reused.

LIME generation — swinb_lora


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


   swinb_lora: 0 missing (all LoRA/adapter prefixes ✅), 0 unexpected
✅ swinb_lora loaded from swinb_lora_finetuned.pt
  [100] checkpoint saved (0 new / 100 reused)...
  [200] checkpoint saved (0 new / 200 reused)...
  [300] checkpoint saved (0 new / 300 reused)...
  [400] checkpoint saved (0 new / 400 reused)...
  [500] checkpoint saved (0 new / 500 reused)...
✅ swinb_lora: 0 new, 502 pkl-reused.

✅ Phase A complete. Manifest rows: 1528


## Cell 12 — Phase B: IG-LIME IoU agreement  (FIX-H: None handling | FIX-J: healthy skip | FIX-K: mask squeeze)

In [13]:
# Separated from Phase A — can be re-run independently after kernel restart.
_p = RESULTS_PATH / 'lime_manifest_partial.csv'

if 'lime_manifest_rows' not in globals() or not lime_manifest_rows:
    assert _p.exists(), '❌ No lime_manifest_rows in memory and no partial CSV. Run Cell 11 first.'
    if _p.stat().st_size > 0:
        lime_manifest_rows = pd.read_csv(str(_p)).to_dict('records')
        print(f'Reloaded lime_manifest from partial CSV: {len(lime_manifest_rows)} rows')
    else:
        lime_manifest_rows = []
        print('No lime_manifest_rows reloaded from empty partial CSV.')

if 'fallback_counter' not in globals():
    fallback_counter = {'count': 0}

ig_lookup = {
    (row['image_id'], row['model']): row['top50_path']
    for _, row in ig_manifest_df[['image_id', 'model', 'top50_path']].drop_duplicates().iterrows()
}

agreement_rows   = []
agreement_partial = RESULTS_PATH / 'lime_ig_agreement_partial.csv'

done_agr_keys = set()
if agreement_partial.exists():
    try:
        _dfp = pd.read_csv(str(agreement_partial))
        agreement_rows  = _dfp.to_dict('records')
        done_agr_keys  |= set(zip(_dfp['image_id'], _dfp['model']))
        print(f'Resumed agreement: {len(agreement_rows)} rows')
    except pd.errors.EmptyDataError:
        print('agreement_partial.csv exists but is empty (EmptyDataError). Starting fresh.')
print(f'Computing IoU for {len(lime_manifest_rows)} manifest rows...')

n_nan_fallback = 0   # FIX-H: LIME returned None (label absent)
n_nan_healthy  = 0   # FIX-J: healthy image with no top50 mask

for i, row in enumerate(lime_manifest_rows):
    image_id   = row['image_id']
    model_name = row['model']
    target_idx = int(row['target_idx'])
    subset     = row['subset']
    key        = (image_id, model_name)

    if key in done_agr_keys:
        continue

    assert key in ig_lookup, f'❌ Missing IG manifest entry for {image_id}/{model_name}'
    top50_path = Path(ig_lookup[key])

    # FIX-J: some test_healthy rows have no top50 mask.
    #   - Healthy images BELOW threshold were skipped entirely in NB04 → no mask file.
    #   - Healthy images ABOVE threshold (Type-A FPs) were processed in NB04 and DO
    #     have a top50 mask; those rows pass the exists() check and continue normally.
    #   exists() gracefully handles both cases without crashing.
    #   Rows without a mask are kept with iou_valid=False for NB07 FP analysis.
    if not top50_path.exists():
        agreement_rows.append({
            'image_id'    : image_id,
            'model'       : model_name,
            'subset'      : subset,
            'target_class': row['target_class'],
            'lime_ig_iou' : float('nan'),
            'iou_valid'   : False,
        })
        done_agr_keys.add(key)
        n_nan_healthy += 1
        continue

    ig_top50_mask = np.load(str(top50_path))

    # FIX-K: NB04 may have saved masks as (224,224,1) or (224,224,3) — squeeze first.
    if ig_top50_mask.ndim == 3:
        ig_top50_mask = ig_top50_mask.squeeze()

    assert ig_top50_mask.shape == (IMG_SIZE, IMG_SIZE), (
        f'❌ Shape mismatch for {image_id}/{model_name}: '
        f'expected ({IMG_SIZE},{IMG_SIZE}), got {ig_top50_mask.shape}'
    )

    lime_pkl_path = Path(row['lime_pkl_path'])
    assert lime_pkl_path.exists(), f'❌ Missing LIME pkl: {lime_pkl_path}. Run Cell 11 first.'
    with open(str(lime_pkl_path), 'rb') as fh:
        explanation = pickle.load(fh)

    # FIX-F note: pkls generated by Cell 11 store explanations with labels=[1]
    # (column 1 of the 2-class output == P(target)). We pass target_idx=1 here.
    lime_mask = get_lime_mask(explanation, target_idx=1, ig_top50_mask=ig_top50_mask)

    # FIX-H: get_lime_mask returns None when label 1 is absent from local_exp
    if lime_mask is None:
        agreement_rows.append({
            'image_id'    : image_id,
            'model'       : model_name,
            'subset'      : subset,
            'target_class': row['target_class'],
            'lime_ig_iou' : float('nan'),
            'iou_valid'   : False,
        })
        done_agr_keys.add(key)
        fallback_counter['count'] += 1
        n_nan_fallback += 1
        continue

    iou_score = compute_iou(ig_top50_mask, lime_mask)
    agreement_rows.append({
        'image_id'    : image_id,
        'model'       : model_name,
        'subset'      : subset,
        'target_class': row['target_class'],
        'lime_ig_iou' : round(iou_score, 6),
        'iou_valid'   : True,
    })
    done_agr_keys.add(key)

    if (i + 1) % 500 == 0:
        pd.DataFrame(agreement_rows).to_csv(str(agreement_partial), index=False)
        print(f'  [{i+1}] agreement checkpoint saved...')

pd.DataFrame(agreement_rows).to_csv(str(agreement_partial), index=False)
print(f'\n✅ Phase B complete. Agreement rows: {len(agreement_rows)}')
print(f'   NaN — no top50 mask (healthy/FP rows): {n_nan_healthy}')
print(f'   NaN — LIME label absent  (FIX-H):      {n_nan_fallback}')
if n_nan_fallback > 0:
    print('   ⚠️  Non-zero LIME fallbacks — check predict_fn or lime version.')

Computing IoU for 1528 manifest rows...
  [500] agreement checkpoint saved...
  [1000] agreement checkpoint saved...
  [1500] agreement checkpoint saved...

✅ Phase B complete. Agreement rows: 1528
   NaN — no top50 mask (healthy/FP rows): 0
   NaN — LIME label absent  (FIX-H):      0


## Cell 13 — Save final manifests and clean up partials  (FIX-L: size-check before unlink)

In [14]:
lime_manifest_df = pd.DataFrame(lime_manifest_rows)
agreement_df     = pd.DataFrame(agreement_rows)

final_lime_manifest = RESULTS_PATH / 'lime_manifest.csv'
final_agreement     = RESULTS_PATH / 'lime_ig_agreement.csv'

lime_manifest_df.to_csv(final_lime_manifest, index=False)
agreement_df.to_csv(final_agreement,         index=False)

# FIX-L: verify final files are non-empty before deleting partials.
#         If Drive disconnected mid-write, to_csv() raises but partials would
#         already be gone in the old code.
for final_path, partial_path in [
    (final_lime_manifest, lime_manifest_partial),
    (final_agreement,     agreement_partial),
]:
    assert final_path.exists() and final_path.stat().st_size > 0, (
        f'❌ Final file empty or missing: {final_path} — NOT deleting partial.'
    )
    if partial_path.exists():
        partial_path.unlink()
        print(f'   Removed partial: {partial_path.name}')

print(f'\n✅ lime_manifest.csv      saved: {len(lime_manifest_df)} rows')
print(f'✅ lime_ig_agreement.csv  saved: {len(agreement_df)} rows')

valid_df = agreement_df[agreement_df['iou_valid'] == True]
print(f'\nMean IG-LIME IoU per model (valid rows only):')
print(valid_df.groupby('model')['lime_ig_iou'].mean().round(4).to_string())
print('\nMean IG-LIME IoU per pathology — all models (valid rows only):')
print(valid_df.groupby('target_class')['lime_ig_iou']
      .mean().sort_values(ascending=False).round(4).to_string())


   Removed partial: lime_manifest_partial.csv
   Removed partial: lime_ig_agreement_partial.csv

✅ lime_manifest.csv      saved: 1528 rows
✅ lime_ig_agreement.csv  saved: 1528 rows

Mean IG-LIME IoU per model (valid rows only):
model
convnextv2_tiny    0.1345
densenet121        0.0855
swinb_lora         0.1898

Mean IG-LIME IoU per pathology — all models (valid rows only):
target_class
Calcification         0.1987
Infiltration          0.1583
Cardiomegaly          0.1506
Aortic enlargement    0.1494
Pleural effusion      0.1420
ILD                   0.1364
Lung Opacity          0.1123
Pulmonary fibrosis    0.1116
Other lesion          0.1016
Nodule/Mass           0.0996
Pleural thickening    0.0954


## Cell 14 — Per-pathology agreement rate summary  (FIX-M: count check | healthy rows excluded from Table 2)

In [15]:
# Re-read from disk — safe after kernel restart
agreement_df = pd.read_csv(RESULTS_PATH / 'lime_ig_agreement.csv')

# Exclude healthy/FP rows (iou_valid=False) for Table 2.
# These rows have no radiologist consensus box — including them would dilute
# the pathology-level faithfulness metrics. They are preserved for NB07 FP analysis.
patho_df = agreement_df[(agreement_df['iou_valid'] == True) & (agreement_df['subset'] == 'patho')].copy()
print(f'Rows used for Table 2 summary: {len(patho_df)} '
      f'(excluded {len(agreement_df) - len(patho_df)} healthy/NaN rows)')

summary_rows = []
for mn in MODEL_NAMES:
    for path_class in sorted(patho_df['target_class'].unique()):
        sub = patho_df[
            (patho_df['model']        == mn) &
            (patho_df['target_class'] == path_class)
        ]
        if sub.empty:
            continue
        mean_iou  = sub['lime_ig_iou'].mean()
        agree_rt  = (sub['lime_ig_iou'] >= AGREEMENT_IOU_THR).mean()
        summary_rows.append({
            'model'            : mn,
            'target_class'     : path_class,
            'n'                : len(sub),
            'mean_lime_ig_iou' : round(mean_iou, 4),
            'agree_rate_gte15' : round(agree_rt, 4),
            'flag'             : 'method-sensitive' if agree_rt < LOW_AGREE_THRESH else 'ok',
        })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(RESULTS_PATH / 'lime_agreement_summary.csv', index=False)
print('✅ lime_agreement_summary.csv saved')

flagged = summary_df[summary_df['flag'] == 'method-sensitive']
if flagged.empty:
    print(f'\n✅ All pathology/model combinations meet ≥{LOW_AGREE_THRESH:.0%} agreement rate.')
else:
    print(f'\n⚠️  {len(flagged)} method-sensitive combinations (agree_rate < {LOW_AGREE_THRESH:.0%}):')
    print(flagged[['model', 'target_class', 'n', 'mean_lime_ig_iou', 'agree_rate_gte15']]
          .to_string(index=False))
    print('   → Flag these in Discussion: Acknowledge <10% agreement as evidence of XAI instability.')

print('\nFull summary:')
print(summary_df[['model','target_class','n','mean_lime_ig_iou','agree_rate_gte15','flag']]
      .to_string(index=False))


Rows used for Table 2 summary: 1383 (excluded 145 healthy/NaN rows)
✅ lime_agreement_summary.csv saved

⚠️  23 method-sensitive combinations (agree_rate < 60%):
          model       target_class   n  mean_lime_ig_iou  agree_rate_gte15
    densenet121 Aortic enlargement 225            0.0952            0.1822
    densenet121       Cardiomegaly  73            0.0932            0.1507
    densenet121                ILD   4            0.0700            0.2500
    densenet121       Infiltration   2            0.0867            0.0000
    densenet121       Lung Opacity  12            0.0648            0.0833
    densenet121        Nodule/Mass   8            0.0415            0.0000
    densenet121       Other lesion   5            0.0662            0.2000
    densenet121   Pleural effusion  34            0.1025            0.2059
    densenet121 Pleural thickening  58            0.0734            0.0690
    densenet121 Pulmonary fibrosis  40            0.0531            0.0000
convnextv2_tin

## Cell 15 — Final verification  (FIX-M: pkl count == manifest row count)

In [16]:
print('NB05 Verification')
print('=' * 60)

checks = {}

for fname in ['lime_manifest.csv', 'lime_ig_agreement.csv', 'lime_agreement_summary.csv']:
    checks[fname] = (RESULTS_PATH / fname).exists()
checks['ig_manifest.csv intact'] = (RESULTS_PATH / 'ig_manifest.csv').exists()
checks['lime_maps/ directory']   = LIME_PATH.exists()

# FIX-M: compare pkl count against manifest row count, not just > 0.
#         One pkl file would pass the old "> 0" check even if 499 were missing.
try:
    manifest_check_df = pd.read_csv(RESULTS_PATH / 'lime_manifest.csv')
    expected_counts = manifest_check_df.groupby('model').size().to_dict()
except Exception:
    expected_counts = {mn: -1 for mn in MODEL_NAMES}

pkl_counts = {mn: len(list(LIME_PATH.glob(f'*_{mn}_lime.pkl'))) for mn in MODEL_NAMES}
for mn in MODEL_NAMES:
    expected = expected_counts.get(mn, 0)
    actual   = pkl_counts.get(mn, 0)
    checks[f'{mn} pkl count'] = (actual == expected)
    if actual != expected:
        print(f'   ⚠️  {mn}: {actual} pkls on disk vs {expected} in manifest')

try:
    agr_cols  = set(pd.read_csv(str(RESULTS_PATH / 'lime_ig_agreement.csv')).columns)
    mani_cols = set(pd.read_csv(str(RESULTS_PATH / 'lime_manifest.csv')).columns)
    summ_cols = set(pd.read_csv(str(RESULTS_PATH / 'lime_agreement_summary.csv')).columns)
    assert 'lime_ig_iou'    in agr_cols,  'lime_ig_iou missing from lime_ig_agreement.csv'
    assert 'iou_valid'      in agr_cols,  'iou_valid missing from lime_ig_agreement.csv'
    assert 'target_class'   in agr_cols,  'target_class missing from lime_ig_agreement.csv'
    assert 'model'          in agr_cols,  'model missing from lime_ig_agreement.csv'
    assert 'lime_pkl_path'  in mani_cols, 'lime_pkl_path missing from lime_manifest.csv'
    assert 'agree_rate_gte15' in summ_cols,'agree_rate_gte15 missing from lime_agreement_summary.csv'
    schema_ok = True
except Exception as e:
    schema_ok = False
    checks[f'CSV schemas ❌ ({e})'] = False

checks['CSV schemas valid'] = schema_ok

for name, ok in checks.items():
    icon  = '✅' if ok else '❌'
    extra = ''
    if name.endswith('pkl count'):
        mn    = name[:-len(' pkl count')]
        exp   = expected_counts.get(mn, '?')
        act   = pkl_counts.get(mn, 0)
        extra = f' ({act}/{exp} files)'
    print(f'  {icon} {name}{extra}')

for fname in ['lime_manifest_partial.csv', 'lime_ig_agreement_partial.csv']:
    if (RESULTS_PATH / fname).exists():
        print(f'  ⚠️  Stale partial file: {fname} — run Cell 13 to clean up.')

print()
all_ok = all(v for v in checks.values() if isinstance(v, bool))
if all_ok:
    print('✅ NB05 complete. Ready to run NB06_faithfulness_eval.ipynb')
else:
    failed = [k for k, v in checks.items() if not v]
    print(f'❌ {len(failed)} check(s) failed: {failed}')


NB05 Verification
  ✅ lime_manifest.csv
  ✅ lime_ig_agreement.csv
  ✅ lime_agreement_summary.csv
  ✅ ig_manifest.csv intact
  ✅ lime_maps/ directory
  ✅ densenet121 pkl count (514/514 files)
  ✅ convnextv2_tiny pkl count (512/512 files)
  ✅ swinb_lora pkl count (502/502 files)
  ✅ CSV schemas valid

✅ NB05 complete. Ready to run NB06_faithfulness_eval.ipynb
